# CircuitVision baseline_v1: real CGHD experiment

Runtime → Change runtime type → **T4 GPU**. Run the cells in order.

Everything saves to `MyDrive/CircuitVision/experiments/baseline_v1`, so a disconnect loses nothing.

**Two hard gates stop execution until you approve them:**
- **Gate 1:** dataset audit and rotation sheets.
- **Gate 2:** YOLO label overlays.

"Run all" halts at each gate by design. The baseline configuration is fixed in `circuitvision/train.py` (`BASELINE`); do not change it for this run.

In [ ]:
import os, sys, json, glob, subprocess, datetime
from google.colab import drive
drive.mount('/content/drive')
EXP  = '/content/drive/MyDrive/CircuitVision/experiments/baseline_v1'
DATA = '/content/drive/MyDrive/CircuitVision/data'
for d in (EXP, DATA, f'{EXP}/logs', f'{EXP}/gates'):
    os.makedirs(d, exist_ok=True)

def run(cmd, log):
    """Run a shell command, stream its output, append it to logs/<log>.log, stop on failure."""
    print(f'$ {cmd}')
    with open(f'{EXP}/logs/{log}.log', 'a') as f:
        f.write(f'\n# {datetime.datetime.now().isoformat()}\n$ {cmd}\n')
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, cwd='/content/CircuitVision', env=os.environ.copy())
        for line in p.stdout:
            print(line, end=''); f.write(line)
    if p.wait() != 0:
        raise RuntimeError(f'{log} failed with exit code {p.returncode}; see {EXP}/logs/{log}.log')

In [ ]:
subprocess.run('rm -rf /content/CircuitVision && git clone -q https://github.com/sudo-pranshu/CircuitVision.git /content/CircuitVision',
               shell=True, check=True)
run('pip install -q -e ".[detect]" && apt-get -qq install -y ngspice > /dev/null', 'setup')
# Colab ships several OpenCV packages that share one cv2/ folder; installing
# ultralytics/easyocr can leave it hollow (cv2.HOGDescriptor missing). Pin exactly
# one complete build. Must stay the LAST package step.
run('bash scripts/pin_opencv.sh', 'setup')
run(f'pip freeze > {EXP}/pip_freeze.txt', 'setup')
run(f'git rev-parse HEAD | tee {EXP}/commit.txt', 'setup')
run(f'nvidia-smi --query-gpu=name,memory.total --format=csv | tee {EXP}/gpu.txt', 'setup')
run('python -m pytest -q tests', 'setup')

## Dataset: CGHD, Zenodo record 14042961 (v14, 4.4 GB, CC BY 3.0)
The zip is cached on Drive after the first download and checked against Zenodo's published MD5.

In [ ]:
ZIP = f'{DATA}/cghd-zenodo-14.zip'
if not os.path.exists(ZIP):
    run(f"wget -q -O {ZIP} 'https://zenodo.org/records/14042961/files/cghd-zenodo-14.zip?download=1'", 'dataset')
run(f'md5sum {ZIP} | tee {EXP}/dataset_md5.txt', 'dataset')
got = open(f'{EXP}/dataset_md5.txt').read().split()[0]
assert got == 'ab1cde6feb5edaafbde711cd2059a2f4', f'MD5 mismatch ({got}): wrong or corrupted download'
run('rm -rf /content/cghd && unzip -q -o ' + ZIP + ' -d /content/cghd', 'dataset')
CGHD = os.path.dirname(sorted(glob.glob('/content/cghd/**/drafter_1', recursive=True))[0])
print('CGHD root:', CGHD)

## Step 1: dataset audit
This reads every XML and image and assumes nothing from the docs. Expect about 10–15 minutes, since it reads each image twice to detect EXIF rotation.

In [ ]:
run(f'python -m circuitvision.audit --cghd {CGHD} --out {EXP}/audit', 'audit')
first = sorted(glob.glob(f'{CGHD}/drafter_1/annotations/*.xml'))[0]
raw = open(first).read()
open(f'{EXP}/audit/raw_xml_full_example.xml', 'w').write(raw)
print(f'--- full raw XML: {first}\n', raw[:4000])
print('--- polarized-symbol objects:\n', open(f'{EXP}/audit/raw_xml_samples.txt').read()[:4000])

## Step 2: rotation sheets
Each sheet has one row per `<rotation>` value, with crops shown including wire context.

In [ ]:
run(f'python -m circuitvision.visualize rotation --cghd {CGHD} --out {EXP}/rotation_check', 'rotation_check')
from IPython.display import Image, display
for p in sorted(glob.glob(f'{EXP}/rotation_check/*.png')):
    print(p); display(Image(p))

## GATE 1: human approval required

Read `audit/audit.md`, `audit/raw_xml_full_example.xml` and the rotation sheets, then fill in the cell below. Answer these from the real examples:

- **`PLUS_AT_0`:** at `rotation=0`, which side of the box holds the diode anode (the flat side of the triangle) or the source's +? Answer `left`, `right`, `top` or `bottom`.
- **`CGHD_CLOCKWISE`:** going from `rotation=0` to `rotation=90`, does the symbol turn clockwise (`True`) or counter-clockwise (`False`)?

If the sheets don't allow a confident answer (missing, inconsistent, or not multiples of 90), leave both as `None`. Orientation is then reported as UNTESTED.

Also send Claude the rotation sheets and `audit.md`.

In [ ]:
GATE1_APPROVED = False     # set True only after inspecting the real sheets
PLUS_AT_0      = None      # 'left' | 'right' | 'top' | 'bottom' | None
CGHD_CLOCKWISE = None      # True | False | None
GATE1_NOTES    = ''        # what you saw, in one or two sentences

assert GATE1_APPROVED, 'GATE 1: inspect audit.md and the rotation sheets, then set GATE1_APPROVED = True'
ORIENT = PLUS_AT_0 in ('left', 'right', 'top', 'bottom') and CGHD_CLOCKWISE in (True, False)
if ORIENT:
    os.environ['CIRCUITVISION_PLUS_AT_0'] = PLUS_AT_0
    os.environ['CIRCUITVISION_CGHD_CLOCKWISE'] = '1' if CGHD_CLOCKWISE else '0'
json.dump({'approved': True, 'plus_at_0': PLUS_AT_0, 'cghd_clockwise': CGHD_CLOCKWISE,
           'orientation_evaluated': ORIENT, 'notes': GATE1_NOTES,
           'time': datetime.datetime.now().isoformat()}, open(f'{EXP}/gates/gate1.json', 'w'), indent=1)
print('orientation stage will run' if ORIENT else 'orientation stage will be reported as UNTESTED')

## Step 3: CGHD → YOLO conversion and label overlays
The split is by drafter: validation = drafter_11, test = drafter_12, everything else is training.

In [ ]:
run(f'python -m circuitvision.cghd_to_yolo --cghd {CGHD} --out /content/yolo --val-drafters 11 --test-drafters 12', 'convert')
run(f'cp /content/yolo/conversion_stats.json {EXP}/', 'convert')
run(f'python -m circuitvision.visualize labels --yolo /content/yolo --out {EXP}/label_check -n 24', 'label_check')
print(open(f'{EXP}/label_check/index.txt').read())
for p in sorted(glob.glob(f'{EXP}/label_check/*.jpg'))[:8]:
    print(p); display(Image(p, width=900))

## GATE 2: human approval required

Look through all 24 overlays in `label_check/`, not just the 8 shown above. Every box must sit on its symbol. A consistent shift, a scale error or a 90° rotation means an EXIF or size problem: **do not approve**; send Claude the overlays and `audit.md` instead.

Pay particular attention to: resistor, capacitor, inductor, voltage source, diode, ground, junction, crossover and text.

In [ ]:
GATE2_APPROVED = False     # set True only if the labels are visibly correct
GATE2_NOTES    = ''

assert GATE2_APPROVED, 'GATE 2: inspect label_check/, then set GATE2_APPROVED = True'
json.dump({'approved': True, 'notes': GATE2_NOTES, 'time': datetime.datetime.now().isoformat()},
          open(f'{EXP}/gates/gate2.json', 'w'), indent=1)

## Step 4: train the baseline YOLOv8s (roughly 3–5 h on a T4)
If Colab disconnects, re-run the setup cells, the dataset cell, the two gate cells (already approved) and the conversion cell, then run the **resume** cell instead of this one.

In [ ]:
run('python scripts/check_opencv.py', 'train')
run(f'python -m circuitvision.train --data /content/yolo/dataset.yaml --exp {EXP}', 'train')

In [ ]:
# Resume after a disconnect (uses the same run and config):
# run(f'python -m circuitvision.train --data /content/yolo/dataset.yaml --exp {EXP} --resume', 'train')

## Step 5: test drafter 12, with metrics and qualitative failures

In [ ]:
run('python scripts/check_opencv.py', 'test')
BEST = f'{EXP}/runs/train/weights/best.pt'
run(f'python -m circuitvision.detector_eval --weights {BEST} --data /content/yolo/dataset.yaml --exp {EXP}', 'test')

## Step 6: downstream stages, ground-truth boxes and detector boxes (reported separately)

In [ ]:
run('python scripts/check_opencv.py', 'downstream')
if ORIENT:
    run(f'python -m circuitvision.orientation --cghd {CGHD} --out {EXP}/models --test-drafters 12 --exp {EXP}', 'orientation')
orient_arg = f'--orient {EXP}/models' if ORIENT else ''
run(f'python -m circuitvision.evaluate --cghd {CGHD} --drafters 12 --exp {EXP} {orient_arg} --ocr --weights {BEST}', 'downstream')

## Evidence bundle
This collects configuration, gates, logs, metrics, CSVs and plots, and leaves out CGHD, checkpoints and training caches.

In [ ]:
readme = f'''# baseline_v1 evidence (REAL CGHD)

commit: {open(f"{EXP}/commit.txt").read().strip()}
dataset md5: {open(f"{EXP}/dataset_md5.txt").read().split()[0]}
gpu: {open(f"{EXP}/gpu.txt").read().strip().splitlines()[-1]}
bundled: {datetime.datetime.now().isoformat()}

audit/            Step 1: dataset audit, raw XML
rotation_check/   Step 2: rotation sheets (Gate 1)
gates/            gate1.json, gate2.json: human approvals and notes
conversion_stats.json, label_check/   Step 3 (Gate 2)
train_config.json, args.yaml, results.csv, results.png   Step 4
test/             Step 5: metrics.json, confusion matrix, per_image.csv, qualitative/
orientation.json  Step 6 orientation (only if Gate 1 fixed the convention)
downstream/       Step 6: gt_boxes.csv, detector.csv, summary.json (reported separately)
logs/             full stdout/stderr of every step
pip_freeze.txt    exact package versions (OpenCV pinned by scripts/pin_opencv.sh)
'''
open(f'{EXP}/EVIDENCE_README.md', 'w').write(readme)
run(f"cd {EXP} && rm -f /content/baseline_v1_evidence.zip && zip -q -r /content/baseline_v1_evidence.zip . "
    "-x 'runs/*' -x 'models/*' -x '*.pt' -x '*.cache'", 'evidence')
for f in ('test/metrics.json', 'downstream/summary.json'):
    print(f'--- {f}'); print(open(f'{EXP}/{f}').read()[:3000])
run('ls -lh /content/baseline_v1_evidence.zip', 'evidence')
from google.colab import files; files.download('/content/baseline_v1_evidence.zip')